In [5]:
ls

evaluation_exp.ipynb   smart_sladding_ml/
evaluation_main.ipynb  valideringssett/


In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## LAST NED ALLE DOKUMENTENE

In [7]:
df_doc = pd.read_csv('valideringssett/bestilling_tinglyst_dokument.csv')

df_doc.head()

,288dee9c-a482-40a8-bbaf-e9b0ae24ce1c,1980,14847,101
0,48e7c584-e2a9-4833-92c1-10d81587dde5,1980,118,63
1,878ba338-ce63-4554-90c4-678ecb64f78e,1980,4201,22
2,f01e080f-0186-4cac-9cb9-d2ee3fe26587,1980,2377,31
3,55cdc87d-a58d-4859-8273-8f33d3a60755,1980,410038,90
4,73a68064-e769-4857-9343-9b2734314347,1980,1349,50


In [ ]:
df_doc["dokument_nr_embete"] = df_doc["1980"].astype(str) + "_" + df_doc["14847"].astype(str) + "_" + df_doc["101"].astype(str)

df_doc.head()

In [ ]:
#URL: https://dokumentbestilling-smart-sladding-manual.atkv3-dev.kartverket-intern.cloud/pantebok/dokument_ident.pdf

import requests
import os

def download_and_save_pdf(url, filename):
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(filename, 'wb') as file:
            file.write(response.content)
        print("PDF downloaded successfully.")
    else:
        print("Failed to download file. HTTP Status Code:", response.status_code)

os.mkdir("valideringssett/dokumenter")
for doc in df_doc["dokument_nr_embete"].unique():
    url = f"https://dokumentbestilling-smart-sladding-manual.atkv3-dev.kartverket-intern.cloud/pantebok/{doc}.pdf"
    filename = f"dokumenter/{doc}.pdf"
    download_and_save_pdf(url, filename)

In [ ]:
doc = "1980_14847_101"
url = f"https://dokumentbestilling-smart-sladding-manual.atkv3-dev.kartverket-intern.cloud/pantebok/{doc}.pdf"
filename = f"valideringssett/dokumenter/{doc}.pdf"
download_and_save_pdf(url, filename)

## STUKTURER LABELS

In [8]:
#Read csv file into a pandas dataframe

df = pd.read_csv('valideringssett/labels.csv')

df.head()

,dokument_aar,dokument_nr,embete,sidetall,index,type,height,width,x,y
0,2017,52610,201,1,0,PERSONNUMMER,11.000000,29.500000,325.000000,211.500000
1,1980,14847,101,1,0,PERSONNUMMER,17.223385,46.110317,534.443991,303.888723
2,1980,14847,101,1,1,PERSONNUMMER,13.888889,43.333333,529.444444,408.888889
3,1980,2377,31,1,0,PERSONNUMMER,15.000000,51.666667,339.444444,363.888889
4,1980,2377,31,1,1,PERSONNUMMER,11.666667,37.222464,435.555192,292.777686


In [9]:
#Create a new row named "dokument_nr_embete" and fill it with a string that contains the values of "dokument_nr" and "embete" columns

df["dokument_nr_embete"] = df["dokument_aar"].astype(str) + "_" + df["dokument_nr"].astype(str) + "_" + df["embete"].astype(str)

df.head()

,dokument_aar,dokument_nr,embete,sidetall,index,type,height,width,x,y,dokument_nr_embete
0,2017,52610,201,1,0,PERSONNUMMER,11.000000,29.500000,325.000000,211.500000,2017_52610_201
1,1980,14847,101,1,0,PERSONNUMMER,17.223385,46.110317,534.443991,303.888723,1980_14847_101
2,1980,14847,101,1,1,PERSONNUMMER,13.888889,43.333333,529.444444,408.888889,1980_14847_101
3,1980,2377,31,1,0,PERSONNUMMER,15.000000,51.666667,339.444444,363.888889,1980_2377_31
4,1980,2377,31,1,1,PERSONNUMMER,11.666667,37.222464,435.555192,292.777686,1980_2377_31


In [10]:
#Collect all rows with "dokument_nr_embete" = "2023_73413_200"
df_2023_73413_200 = df[df["dokument_nr_embete"] == "2023_73413_200"]
print(df_2023_73413_200)

      dokument_aar  dokument_nr  embete  sidetall  index          type  \
1351          2023        73413     200         1      0  PERSONNUMMER   
1352          2023        73413     200         1      1  PERSONNUMMER   
1353          2023        73413     200         1      2  PERSONNUMMER   
1354          2023        73413     200         1      3  PERSONNUMMER   
1355          2023        73413     200         1      4  PERSONNUMMER   

         height      width           x           y dokument_nr_embete  
1351  10.555556  25.555556  403.888889  277.777778     2023_73413_200  
1352  11.111111  26.666667  207.222222  541.666667     2023_73413_200  
1353  11.666667  26.666667  535.555556  540.555556     2023_73413_200  
1354  10.000000  26.111111   76.111111  637.777778     2023_73413_200  
1355  10.000000  25.000000   76.111111  685.555556     2023_73413_200  


In [ ]:
#Create a new dataframe that contains only the row dokument_nr_embete from df_doc

df_labels = pd.DataFrame(df_doc["dokument_nr_embete"])

print(len(df_labels))

df_labels.head()

In [ ]:
# Assuming df is your main DataFrame containing all bounding boxes
def organize_bounding_boxes(df):
    results = []

    # Group by document number
    grouped_docs = df.groupby('dokument_nr_embete')

    for doc_id, doc_data in grouped_docs:
        max_page = doc_data['sidetall'].max()  # Determine the highest page number
        bb_per_page = [[] for _ in range(max_page)]  # Prepare a list of lists for each page

        # Group by page within each document
        grouped_pages = doc_data.groupby('sidetall')
        for page_number, page_data in grouped_pages:
            # Adjust for zero-based index if necessary
            page_index = page_number - 1
            # Collect all bounding boxes for the page
            bb_per_page[page_index] = page_data[['height', 'width', 'x', 'y']].values.tolist()

        results.append({
            'dokument_nr_embete': doc_id,
            'bounding_boxes': bb_per_page
        })

    return results

organized_data = organize_bounding_boxes(df)

#Save the organized data into a csv file

for doc in df_labels["dokument_nr_embete"]:
    if doc not in [x['dokument_nr_embete'] for x in organized_data]:
        organized_data.append({
            'dokument_nr_embete': doc,
            'bounding_boxes': []
        })

print(len(organized_data))

df_organized = pd.DataFrame(organized_data)

print(df_organized.head())

df_organized.to_csv('valideringssett/organized_data.csv', index=False)


In [ ]:
from pdf2image import convert_from_path
import PyPDF2

def get_pdf_dimensions(pdf_path):
    pdf = PyPDF2.PdfReader(open(pdf_path, "rb"))
    dimensions = []
    for page in range(len(pdf.pages)):
        media_box = pdf.pages[page].mediabox
        width = float(media_box.width)
        height = float(media_box.height)
        dimensions.append((width, height))
    return dimensions

labels_df = pd.read_csv("valideringssett/organized_data.csv")

def get_images_and_bb(labels_df, idx):
    row = labels_df.iloc[idx]
    doc = row["dokument_nr_embete"]
    bbs = row["bounding_boxes"]
    bbs = eval(bbs)
    print(doc)
    filename = f"valideringssett/dokumenter/{doc}.pdf"

    dimensions = get_pdf_dimensions(filename)
    images = []
    for page_number, (width, height) in enumerate(dimensions):
        # Convert page to image
        temp_images = convert_from_path(filename, first_page=page_number+1, last_page=page_number+1, size=(int(width), int(height)))
        images.extend(temp_images)  # Extend the list with the new images
    return images, bbs

In [ ]:

def visualize_bounding_boxes(df, idx):
    images, bbs = get_images_and_bb(df, idx)
    print(bbs)
    if bbs == []:
        print("No bounding boxes for this document")
        return
    for i, image in enumerate(images):
        if i >= len(bbs):
            continue
        fig, ax = plt.subplots(figsize=(20, 20))
        #Display image
        ax.imshow(image)
        #Iterate over all bounding boxes
        for bb in bbs[i]:
            #Create a rectangle patch
            rect = plt.Rectangle((bb[2], bb[3]), bb[1], bb[0], edgecolor='r', facecolor="none")
            #Add the rectangle to the axes
            ax.add_patch(rect)

visualize_bounding_boxes(labels_df, 596)
